# Лабораторная работа № 4. Регистр сдвига с линейной обратной связью

**Вариант № 10.** В файле «ММБИС Варианты для лаб 4.xlsx» даны длина регистра $L$ и первые $2L$ выходных битов. Нужно составить и решить систему над $GF(2)$, восстановить полином и начальное состояние, затем вычислить следующие десять битов.

При запуске из корня проекта длина регистра и биты загружаются из `data/lab_04_lfsr.json`.

## 1. Система уравнений

Используем обозначения со слайдов 4–6 файла «LFSR.pptx». Для каждого $n=L,\ldots,2L-1$ записываем
$$
s_n=c_1s_{n-1}\oplus c_2s_{n-2}\oplus\cdots\oplus c_Ls_{n-L}.
$$
Коэффициенты $c_1,\ldots,c_L$ неизвестны. Предыдущие $L$ битов в обратном хронологическом порядке образуют строку матрицы, а $s_n$ становится правой частью.

In [1]:
import json
from pathlib import Path

inputs = json.loads(Path('data/lab_04_lfsr.json').read_text(encoding='utf-8'))
L = inputs['register_length']
observed = [int(bit) for bit in inputs['observed_bits']]
assert len(observed) == 2 * L and set(observed) <= {0, 1}

augmented = [
    [observed[n - lag] for lag in range(1, L + 1)] + [observed[n]]
    for n in range(L, 2 * L)
]
print('Наблюдения —', ''.join(map(str, observed)))
print('Расширенная матрица [A | b]')
for row in augmented:
    print(*row[:-1], '|', row[-1])

Наблюдения — 111100001111111010
Расширенная матрица [A | b]
1 0 0 0 0 1 1 1 1 | 1
1 1 0 0 0 0 1 1 1 | 1
1 1 1 0 0 0 0 1 1 | 1
1 1 1 1 0 0 0 0 1 | 1
1 1 1 1 1 0 0 0 0 | 1
1 1 1 1 1 1 0 0 0 | 1
1 1 1 1 1 1 1 0 0 | 0
0 1 1 1 1 1 1 1 0 | 1
1 0 1 1 1 1 1 1 1 | 0


## 2. Метод Гаусса над $GF(2)$

В каждом столбце ищем опорную единицу, меняем строки местами и устраняем единицы ниже опорной. Сложение строк выполняется покомпонентным XOR. На обратном ходе подставляем найденные переменные, начиная с последней опорной строки.

In [2]:
def solve_gf2(augmented):
    matrix = [row.copy() for row in augmented]
    variable_count = len(matrix[0]) - 1
    pivots = []

    for column in range(variable_count):
        pivot_row = len(pivots)
        chosen = next((r for r in range(pivot_row, len(matrix))
                       if matrix[r][column]), None)
        if chosen is None:
            continue
        matrix[pivot_row], matrix[chosen] = matrix[chosen], matrix[pivot_row]

        for r in range(pivot_row + 1, len(matrix)):
            if matrix[r][column]:
                matrix[r] = [a ^ b for a, b in zip(matrix[r], matrix[pivot_row])]
        pivots.append(column)

    if any(not any(row[:-1]) and row[-1] for row in matrix):
        raise ValueError('Система несовместна')
    if len(pivots) < variable_count:
        raise ValueError('Коэффициенты не определяются однозначно')

    solution = [0] * variable_count
    for r, column in reversed(list(enumerate(pivots))):
        bit = matrix[r][-1]
        for j in range(column + 1, variable_count):
            bit ^= matrix[r][j] & solution[j]
        solution[column] = bit

    return solution, matrix, len(pivots)

coefficients, echelon, rank = solve_gf2(augmented)
print('Ступенчатая форма')
for row in echelon:
    print(*row[:-1], '|', row[-1])
print('Ранг —', rank)
print('Коэффициенты c₁…cₗ —', coefficients)
assert all(sum(a * c for a, c in zip(row[:-1], coefficients)) % 2 == row[-1]
           for row in augmented)

Ступенчатая форма
1 0 0 0 0 1 1 1 1 | 1
0 1 0 0 0 1 0 0 0 | 0
0 0 1 0 0 0 1 0 0 | 0
0 0 0 1 0 0 0 1 0 | 0
0 0 0 0 1 0 0 0 1 | 0
0 0 0 0 0 1 0 0 0 | 0
0 0 0 0 0 0 1 0 0 | 1
0 0 0 0 0 0 0 1 1 | 0
0 0 0 0 0 0 0 0 1 | 1
Ранг — 9
Коэффициенты c₁…cₗ — [0, 0, 1, 1, 1, 0, 1, 1, 1]


## 3. Полином, начальное состояние и продолжение

По соглашению методички строим $P(x)=1+\sum_{i=1}^Lc_ix^i$. При правом выходе регистра первые $L$ битов считываются от правого края к левому. Поэтому начальное состояние слева направо записывается в обратном порядке относительно первых $L$ выходных битов. Воспроизводим наблюдение из этих первых $L$ битов и затем продолжаем рекурсию ещё на десять шагов.

In [3]:
def extend_sequence(initial_outputs, coefficients, count):
    sequence = list(initial_outputs)
    for _ in range(count):
        next_bit = 0
        for lag, coefficient in enumerate(coefficients, start=1):
            next_bit ^= coefficient & sequence[-lag]
        sequence.append(next_bit)
    return sequence

def bits_text(bits):
    return ''.join(map(str, bits))

initial_outputs = observed[:L]
seed_left_to_right = initial_outputs[::-1]
terms = ['1']
for power, coefficient in enumerate(coefficients, start=1):
    if coefficient:
        terms.append('x' if power == 1 else f'x^{power}')

extended = extend_sequence(initial_outputs, coefficients, L + 10)
assert extended[:2 * L] == observed
next_ten = extended[2 * L:]
assert len(next_ten) == 10

print('P(x) =', ' + '.join(terms))
print('Первые L выходных битов —', bits_text(initial_outputs))
print('Seed слева направо —', bits_text(seed_left_to_right))
print('Воспроизведённые 2L битов —', bits_text(extended[:2 * L]))
print('Следующие 10 битов —', bits_text(next_ten))

P(x) = 1 + x^3 + x^4 + x^5 + x^7 + x^8 + x^9
Первые L выходных битов — 111100001
Seed слева направо — 100001111
Воспроизведённые 2L битов — 111100001111111010
Следующие 10 битов — 1101001111


## Вывод

Ранг системы равен 9, поэтому коэффициенты определяются однозначно. Полином $P(x)=1+x^3+x^4+x^5+x^7+x^8+x^9$ и начальное состояние `100001111` слева направо воспроизводят все 18 наблюдаемых битов. Следующие десять битов — `1101001111`.